# FINA 60201A - TP 2 Submission Notebook

This notebook is the standalone submission version of the project. It mirrors the written report and embeds the project code directly so the notebook can be read on its own.

Notes:
- WRDS-dependent cells still require valid school access.
- Generated `data/` files are not committed to the repository, so rerunning the workflows will regenerate them locally.
- The original assignment and theory PDFs are stored in `docs/`.


## How To Use This Notebook

1. Read the report sections below for the written answers.
2. Review the embedded code sections for the full implementation.
3. If needed, run the helper cells at the end to regenerate Question 1, Part One, or Part Two.


# FINA 60201A - TP2


## Part One


### Question 1

The assignment asks for the quarterly series of Johnson & Johnson's debt in current liabilities (`DLCQ_t`) and long-term debt (`DLTTQ_t`) from 2024Q4 to 2025Q4, then for the daily series

```text
L_t = DLCQ_t + 0.5 * DLTTQ_t
```

I downloaded the data from WRDS using the Compustat Fundamentals Quarterly table (`comp.fundq`) for Johnson & Johnson (`GVKEY 006266`). The fields used are the exact assignment fields:

- `DLCQ_t = dlcq`
- `DLTTQ_t = dlttq`

Compustat reports these debt values in millions of USD, so I converted them to USD in the processing pipeline before computing `L_t`.

Quarterly observations obtained for Johnson & Johnson:

| Quarter | Statement date | `DLCQ_t` ($bn) | `DLTTQ_t` ($bn) | `L_t` ($bn) |
| --- | --- | ---: | ---: | ---: |
| 2024Q4 | 2024-12-31 | 5.983 | 30.651 | 21.308 |
| 2025Q1 | 2025-03-31 | 13.897 | 38.355 | 33.075 |
| 2025Q2 | 2025-06-30 | 11.526 | 39.235 | 31.143 |
| 2025Q3 | 2025-09-30 | 6.387 | 39.408 | 26.091 |
| 2025Q4 | 2025-12-31 | 8.495 | 39.438 | 28.214 |

Daily interpolation method:

- I used linear interpolation on calendar days between consecutive quarter-end statement dates.
- This produces a daily series from `2024-12-31` to `2025-12-31`, for a total of 366 observations.

Generated files:

- WRDS Compustat extract: `data/raw/compustat_jnj_debt_quarterly.csv`
- Quarterly source table: `data/raw/question1_jnj_quarterly_debt_series.csv`
- Daily interpolated series: `data/processed/question1_jnj_daily_default_point.csv`


### Question 2

I downloaded the daily equity series from WRDS using the CRSP daily stock file v2 (`crsp.dsf_v2`) for Johnson & Johnson (`PERMCO 21018`, `PERMNO 22111`). The daily market capitalization is computed as:

```text
Market Cap_t = |Price_t| * Shares Outstanding_t
```

In CRSP v2 this is equivalent to `dlycap * 1000`, because `dlycap` is reported in thousands of USD.

Question 2 output summary:

- Observation window: `2025-01-02` to `2025-12-31`
- Number of daily observations: `250`
- First market capitalization: `346.746` billion USD
- Last market capitalization: `498.604` billion USD
- Minimum market capitalization in 2025: `342.027` billion USD
- Maximum market capitalization in 2025: `515.999` billion USD

Generated file:

- CRSP market cap series: `data/raw/part1_q2_jnj_market_cap_daily.csv`


### Question 3

For the daily risk-free rate, I used the St. Louis Fed one-year zero-coupon series `THREEFY1`. I aligned this series with the CRSP market capitalization dates and with the daily default-point series from Question 1. The merged sample contains `248` common dates from `2025-01-02` to `2025-12-31`.

I then applied the standard iterative Merton procedure:

1. Start from an initial guess for asset volatility.
2. For each day, solve the Black-Scholes equity equation for the asset value.
3. Re-estimate asset volatility from the resulting daily asset returns.
4. Iterate until convergence.

Result:

- Convergence achieved in `2` iterations
- Estimated annual asset volatility at end-2025: `18.477%`
- Estimated asset value at `2025-12-31`: `525.845` billion USD

Generated files:

- FRED one-year zero-coupon series: `data/raw/part1_q3_fred_1y_zero_coupon_daily.csv`
- Aligned daily estimation inputs: `data/processed/part1_q3_aligned_inputs.csv`
- Estimated daily asset series: `data/processed/part1_q3_asset_series.csv`
- Asset-value summary: `data/processed/part1_q3_asset_summary.csv`


### Question 4

Using the Question 1 end-2025 values, I imposed:

- one-year default point = `L_t = 28.214` billion USD
- twenty-year default point = `DLTTQ_t = 39.438` billion USD

I then linearly inter- and extrapolated the default point over maturities from `0` to `30` years.

Key points on the default frontier:

- `D(0) = 27.623` billion USD
- `D(1) = 28.214` billion USD
- `D(20) = 39.438` billion USD
- `D(30) = 45.345` billion USD

Generated file:

- Default frontier: `data/processed/part1_q4_default_frontier.csv`


### Question 5

The first trading day of 2026 in the risk-free dataset is `2026-01-02`. For Question 5, the practical challenge is that the St. Louis Fed/FRED fitted zero-coupon series are directly accessible for benchmark maturities such as `1Y`, `5Y`, and `10Y`, while the complete daily `1Y-30Y` term structure needed for pricing is available from the underlying Federal Reserve Board nominal yield curve table (`SVENY01` to `SVENY30`).

To make the source choice transparent, I validated the `2026-01-02` Board curve against the St. Louis Fed/FRED fitted zero-coupon yields at overlapping benchmark maturities:

| Maturity | St. Louis Fed / FRED (%) | Board curve (%) | Difference (bps) |
| --- | ---: | ---: | ---: |
| 1Y | 3.5075 | 3.4986 | 0.8900 |
| 5Y | 3.7151 | 3.7178 | -0.2700 |
| 10Y | 4.2640 | 4.2822 | -1.8200 |

The overlap differences stay within about `1.82` basis points, so the full Board curve is a very tight proxy for the St. Louis Fed fitted zero-coupon release family when extending the pricing curve out to `30` years.

Using:

- the end-2025 asset value from Question 3,
- the end-2025 asset volatility from Question 3,
- the default frontier from Question 4,
- and the `2026-01-02` zero-coupon risk-free curve,

I priced risky zero-coupon debt under the Merton model and converted the prices into risky yields and credit spreads.

Selected maturities:

| Maturity | Risk-free yield (%) | Risky yield (%) | Credit spread (bps) |
| --- | ---: | ---: | ---: |
| 1Y | 3.4986 | 3.4986 | 0.0000 |
| 5Y | 3.7178 | 3.7178 | 0.0000 |
| 10Y | 4.2822 | 4.2822 | 0.0000 |
| 20Y | 4.9458 | 4.9458 | 0.0036 |
| 30Y | 5.1050 | 5.1052 | 0.0209 |

The Merton-implied term structure is therefore extremely tight for Johnson & Johnson, which is consistent with the large asset cushion relative to the default frontier.

Generated files:

- St. Louis Fed benchmark curve used for the Q5 source check: `data/raw/part1_q5_fred_zero_coupon_benchmarks_daily.csv`
- Full daily zero-coupon curve used for Question 5: `data/raw/part1_zero_coupon_yield_curve_daily.csv`
- Q5 risk-free source validation: `data/processed/part1_q5_risk_free_source_validation.csv`
- Credit spread term structure: `data/processed/part1_q5_credit_spread_term_structure.csv`


## Part Two


### Question 1

The assignment asks for the risk-free zero-coupon term structure on `2026-02-27`, the associated AA curve for the 11 given maturities, and the resulting credit spreads. The assignment wording references the St. Louis Fed fitted zero-coupon curve family; for the actual Nelson-Siegel-Svensson (`NSS`) parameter row, I used the Federal Reserve Board daily nominal yield-curve table because it publishes the `BETA0`, `BETA1`, `BETA2`, `BETA3`, `TAU1`, and `TAU2` parameters directly for each date.

Estimated `2026-02-27` NSS parameters:

| Date | `BETA0` | `BETA1` | `BETA2` | `BETA3` | `TAU1` | `TAU2` |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 2026-02-27 | 1.615248 | 2.172234 | -0.000000 | 10.475890 | 1.459293 | 16.868434 |

Using these parameters, I computed the risk-free zero-coupon yields

```text
zcy(t) = beta0
       + beta1 * ((1 - e^(-t/tau1)) / (t/tau1))
       + beta2 * (((1 - e^(-t/tau1)) / (t/tau1)) - e^(-t/tau1))
       + beta3 * (((1 - e^(-t/tau2)) / (t/tau2)) - e^(-t/tau2))
```

for the 11 required maturities, then formed credit spreads as `AA(t) - zcy(t)`.

| Maturity | Risk-free (%) | AA (%) | Credit spread (bps) |
| --- | ---: | ---: | ---: |
| 0.5Y | 3.6067 | 3.7600 | 15.33 |
| 1Y | 3.4862 | 3.7900 | 30.38 |
| 2Y | 3.3717 | 3.8600 | 48.83 |
| 3Y | 3.3648 | 3.9600 | 59.52 |
| 4Y | 3.4187 | 4.0800 | 66.13 |
| 5Y | 3.5060 | 4.2000 | 69.40 |
| 7Y | 3.7207 | 4.5200 | 79.93 |
| 10Y | 4.0444 | 4.8300 | 78.56 |
| 15Y | 4.4605 | 5.4200 | 95.95 |
| 20Y | 4.7087 | 5.6900 | 98.13 |
| 30Y | 4.8471 | 6.0100 | 116.29 |

The AA credit spread curve is therefore upward sloping overall, rising from about `15.33` bps at `6M` to about `116.29` bps at `30Y`.

To make the St. Louis Fed source wording explicit, I validated the `2026-02-27` FRB/NSS-implied yields against the overlapping St. Louis Fed / FRED fitted zero-coupon benchmark maturities:

| Maturity | St. Louis Fed / FRED (%) | Board NSS (%) | Difference (bps) |
| --- | ---: | ---: | ---: |
| 1Y | 3.4742 | 3.4862 | -1.20 |
| 5Y | 3.5342 | 3.5060 | 2.82 |
| 10Y | 4.0481 | 4.0444 | 0.37 |

The overlap differences stay within about `2.82` basis points, so the FRB parameter row is a very tight implementation source for the St. Louis Fed fitted zero-coupon term structure on the assignment date.

Generated files:

- NSS parameter row: `data/raw/part2_q1_nss_params_2026-02-27.csv`
- Observed AA yields: `data/raw/part2_q1_aa_yields_observed.csv`
- St. Louis Fed benchmark yields for source validation: `data/raw/part2_q1_fred_zero_coupon_benchmarks_daily.csv`
- Question 1 term structures: `data/processed/part2_q1_term_structures.csv`
- Question 1 source validation: `data/processed/part2_q1_source_validation.csv`


### Question 2

The Duffee (1999) specification is

```text
dr_t = kappa_r * (theta_r - r_t) dt + sigma_r * sqrt(r_t) dW_t
ds_t = kappa_s * (theta_s - s_t) dt + sigma_s * sqrt(s_t) dZ_t
lambda_t = alpha + beta * r_t + s_t
```

with `dW_t dZ_t = 0`.

The defaultable zero-coupon price is

```text
P^D(0,t) = E[exp(-integral_0^t (r_u + lambda_u) du)]
         = E[exp(-integral_0^t (alpha + (1 + beta) r_u + s_u) du)]
```

Because `r_t` and `s_t` are independent, define `x_t = (1 + beta) r_t`. Then `x_t` is still a CIR process with:

- the same mean-reversion speed `kappa_r`
- long-run mean `(1 + beta) theta_r`
- volatility `sigma_r * sqrt(1 + beta)`

Therefore

```text
P^D(0,t) = exp(-alpha * t) * P_CIR^x(0,t) * P_CIR^s(0,t)
```

and the defaultable zero-coupon yield is

```text
Y(t) = -(1/t) * ln(P^D(0,t))
     = alpha
       + y_CIR(t; (1 + beta) r_0, kappa_r, (1 + beta) theta_r, sigma_r * sqrt(1 + beta))
       + y_CIR(t; s_0, kappa_s, theta_s, sigma_s)
```

This is the pricing formula I used for the calibration in Question 3 and for the non-callable bond valuation in Question 5.


### Question 3

I calibrated the model in two steps exactly as requested.

Step 1: fit the CIR risk-free term structure to the 11 `2026-02-27` risk-free zero-coupon yields.

Step 2: holding those risk-free parameters fixed, fit the Duffee defaultable-yield curve to the 11 observed AA yields.

Calibrated parameters:

| Block | Parameter | Value |
| --- | --- | ---: |
| Risk-free CIR | `r_0` | 0.032816 |
| Risk-free CIR | `kappa_r` | 0.036991 |
| Risk-free CIR | `theta_r` | 0.077737 |
| Risk-free CIR | `sigma_r` | 0.017775 |
| Risk-free CIR | RMSE | 15.3264 bps |
| Duffee credit block | `s_0` | 0.000563 |
| Duffee credit block | `kappa_s` | 0.125016 |
| Duffee credit block | `theta_s` | 0.012993 |
| Duffee credit block | `sigma_s` | 0.056106 |
| Duffee credit block | `alpha` | 0.002384 |
| Duffee credit block | `beta` | 0.001503 |
| Duffee credit block | RMSE | 8.5387 bps |

Selected fitted values:

| Maturity | Observed AA (%) | Fitted AA (%) | Error (bps) |
| --- | ---: | ---: | ---: |
| 1Y | 3.7900 | 3.7378 | -5.22 |
| 5Y | 4.2000 | 4.2855 | 8.55 |
| 10Y | 4.8300 | 4.8292 | -0.08 |
| 20Y | 5.6900 | 5.6076 | -8.24 |
| 30Y | 6.0100 | 6.1377 | 12.77 |

The credit block fit is materially tighter than the risk-free CIR fit, which is expected because a one-factor CIR curve is only a parsimonious approximation to the `NSS` shape. The fitted Duffee curve remains smooth and economically reasonable from `0` to `30` years.

Generated files:

- Risk-free CIR node fit: `data/processed/part2_q3_risk_free_cir_fit.csv`
- Duffee node fit: `data/processed/part2_q3_duffee_fit.csv`
- Continuous fitted curve from `0` to `30Y`: `data/processed/part2_q3_duffee_curve.csv`
- Calibration parameters: `data/processed/part2_q3_parameters.csv`


### Question 4

I priced the AA callable corporate bond with the required two-dimensional explicit finite-difference method using the state variables `(r_t, s_t)` and the Duffee discount-and-default rate

```text
q(r, s) = alpha + (1 + beta) r + s
```

Bond characteristics:

- Face value: `1,000,000`
- Maturity: `5` years
- Coupon rate: `4%`
- Coupon frequency: semiannual
- Call feature: callable at par at any time

Numerical setup:

- `81 x 81` state grid in `(r, s)`
- `r_max = 20%`
- `s_max = 20%`
- `2,782` time steps
- `dt = 0.001797` years

I imposed the callability cap at every time step. Between coupon dates, the call price is `par = 1,000,000`. On coupon dates, I used the more defensible cum-coupon exercise convention: the issuer can redeem at `par + coupon = 1,020,000`, which is equivalent to paying the current coupon and then calling the bond at par.

Results:

- Callable bond price: `968,946.18` USD
- Matched non-callable bond closed-form price: `986,209.64` USD
- Non-callable finite-difference validation price: `986,112.65` USD
- Finite-difference minus closed-form validation error: `-97.00` USD

The finite-difference validation error is less than `0.01%` of face value, which is sufficiently tight for the assignment.

Generated file:

- Bond valuation summary: `data/processed/part2_q4_bond_valuation.csv`


### Question 5

I computed the bond-equivalent semiannual YTM implied by:

- the callable-bond finite-difference price from Question 4
- the matched non-callable closed-form price from the Duffee term structure

Results:

| Bond | Price (USD) | YTM (%) |
| --- | ---: | ---: |
| Callable | 968,946.18 | 4.7042 |
| Non-callable | 986,209.64 | 4.3095 |

Comparison:

- Callable bond price discount relative to the non-callable bond: `17,263.46` USD
- Callable bond YTM pickup: `39.47` bps

The higher YTM on the callable bond is economically consistent with the embedded issuer call option: the option lowers the bond price and raises the yield required by investors.

Generated file:

- YTM comparison: `data/processed/part2_q5_ytm_comparison.csv`


## Embedded Project Code

The following cells contain the project code exactly as implemented in the repository. Keeping the code here makes the notebook a self-contained submission artifact.

### File: `config/wrds.credentials.example.json`

In [ ]:
{
  "wrds_username": "your_wrds_username",
  "compustat_library": "comp",
  "compustat_table": "fundq",
  "gvkey": "006266",
  "start_date": "2024-12-01",
  "end_date": "2025-12-31"
}


### File: `scripts/fetch_wrds_compustat_q1.py`

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fixed_income_tp2.wrds_compustat import fetch_compustat_jnj_debt


if __name__ == "__main__":
    config_path = PROJECT_ROOT / "config" / "wrds.credentials.json"
    output_path = PROJECT_ROOT / "data" / "raw" / "compustat_jnj_debt_quarterly.csv"

    if not config_path.exists():
        raise SystemExit(
            "Missing config/wrds.credentials.json. Copy config/wrds.credentials.example.json first."
        )

    csv_path = fetch_compustat_jnj_debt(config_path=config_path, output_path=output_path)
    print(f"Saved WRDS Compustat extract to: {csv_path}")


### File: `scripts/run_question1.py`

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fixed_income_tp2.question1 import run_question1


if __name__ == "__main__":
    result = run_question1(PROJECT_ROOT)
    print(f"Question 1 source: {result['source']}")
    print(f"Quarterly output: {result['quarterly_output']}")
    print(f"Daily output: {result['daily_output']}")


### File: `scripts/run_part_one.py`

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fixed_income_tp2.part_one import run_part_one


if __name__ == "__main__":
    result = run_part_one(PROJECT_ROOT)
    for key, value in result.items():
        print(f"{key}: {value}")


### File: `scripts/run_part_two.py`

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fixed_income_tp2.part_two import run_part_two


if __name__ == "__main__":
    result = run_part_two(PROJECT_ROOT)
    for key, value in result.items():
        print(f"{key}: {value}")


### File: `src/fixed_income_tp2/__init__.py`

In [ ]:
"""Utilities for the FINA 60201A TP2 project."""



### File: `src/fixed_income_tp2/wrds_compustat.py`

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path


MIN_SUPPORTED_PYTHON = (3, 8)
MAX_SUPPORTED_PYTHON = (3, 12)


def _load_config(config_path: Path) -> dict[str, str]:
    with config_path.open("r", encoding="utf-8") as handle:
        config = json.load(handle)

    required_keys = {
        "wrds_username",
        "compustat_library",
        "compustat_table",
        "gvkey",
        "start_date",
        "end_date",
    }
    missing = [key for key in sorted(required_keys) if not config.get(key)]
    if missing:
        missing_list = ", ".join(missing)
        raise ValueError(f"{config_path.name} is missing required values: {missing_list}")
    return config


def _assert_supported_python() -> None:
    current = sys.version_info[:2]
    if MIN_SUPPORTED_PYTHON <= current <= MAX_SUPPORTED_PYTHON:
        return

    minimum = ".".join(str(part) for part in MIN_SUPPORTED_PYTHON)
    maximum = ".".join(str(part) for part in MAX_SUPPORTED_PYTHON)
    current_label = ".".join(str(part) for part in current)
    raise RuntimeError(
        "The official WRDS Python package is documented for Python "
        f"{minimum} through {maximum}, but this interpreter is {current_label}. "
        "Use a separate Python 3.12 virtual environment for the WRDS download step."
    )


def fetch_compustat_jnj_debt(config_path: Path, output_path: Path) -> Path:
    _assert_supported_python()

    try:
        import wrds
    except ImportError as exc:
        raise RuntimeError(
            "The wrds package is not installed in this environment. "
            "Create a Python 3.12 virtual environment and run "
            "`python -m pip install -U pip wheel wrds pandas`."
        ) from exc

    config = _load_config(config_path)

    sql = f"""
        select
            gvkey,
            datadate,
            fyearq,
            fqtr,
            dlcq,
            dlttq
        from {config['compustat_library']}.{config['compustat_table']}
        where gvkey = %(gvkey)s
          and datadate between %(start_date)s and %(end_date)s
          and indfmt = 'INDL'
          and datafmt = 'STD'
          and consol = 'C'
          and popsrc = 'D'
        order by datadate
    """

    params = {
        "gvkey": config["gvkey"],
        "start_date": config["start_date"],
        "end_date": config["end_date"],
    }

    with wrds.Connection(wrds_username=config["wrds_username"]) as db:
        frame = db.raw_sql(sql, params=params, date_cols=["datadate"])

    if frame.empty:
        raise RuntimeError(
            "WRDS returned no rows. Check that your account has Compustat access and that the query filters are valid."
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_path, index=False)
    return output_path



### File: `src/fixed_income_tp2/question1.py`

In [ ]:
from __future__ import annotations

import csv
from dataclasses import dataclass
from datetime import date, datetime, timedelta
from pathlib import Path


COMPUSTAT_INPUT_NAME = "compustat_jnj_debt_quarterly.csv"
REQUIRED_QUARTERS = {
    "2024Q4": {"statement_date": date(2024, 12, 31)},
    "2025Q1": {"statement_date": date(2025, 3, 31)},
    "2025Q2": {"statement_date": date(2025, 6, 30)},
    "2025Q3": {"statement_date": date(2025, 9, 30)},
    "2025Q4": {"statement_date": date(2025, 12, 31)},
}


@dataclass(frozen=True)
class QuarterlyDebtObservation:
    quarter: str
    statement_date: date
    dlcq_usd: float
    dlttq_usd: float
    source: str
    field_mapping: str

    @property
    def l_usd(self) -> float:
        return self.dlcq_usd + 0.5 * self.dlttq_usd

    @property
    def dlcq_billion_usd(self) -> float:
        return self.dlcq_usd / 1_000_000_000

    @property
    def dlttq_billion_usd(self) -> float:
        return self.dlttq_usd / 1_000_000_000

    @property
    def l_billion_usd(self) -> float:
        return self.l_usd / 1_000_000_000


def _coerce_float(value: str | None) -> float:
    if value is None:
        raise ValueError("Expected a numeric value, received None.")
    cleaned = value.replace(",", "").replace("$", "").strip()
    if not cleaned:
        raise ValueError("Expected a numeric value, received an empty string.")
    return float(cleaned)


def _quarter_from_date(statement_date: date) -> str:
    month_to_quarter = {3: "Q1", 6: "Q2", 9: "Q3", 12: "Q4"}
    try:
        quarter_suffix = month_to_quarter[statement_date.month]
    except KeyError as exc:
        raise ValueError(f"Unsupported statement month: {statement_date.month}") from exc
    return f"{statement_date.year}{quarter_suffix}"


def _parse_date(value: str) -> date:
    return datetime.strptime(value.strip(), "%Y-%m-%d").date()


def load_compustat_observations(csv_path: Path) -> list[QuarterlyDebtObservation]:
    selected: dict[str, QuarterlyDebtObservation] = {}
    with csv_path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        missing = {"dlcq", "dlttq"} - {name.lower() for name in (reader.fieldnames or [])}
        if missing:
            missing_list = ", ".join(sorted(missing))
            raise ValueError(f"{csv_path.name} is missing required columns: {missing_list}")

        for row in reader:
            normalized = {key.lower(): value for key, value in row.items() if key is not None}
            date_value = normalized.get("statement_date") or normalized.get("datadate") or normalized.get("date")
            quarter_value = normalized.get("quarter")

            if date_value:
                statement_date = _parse_date(date_value)
                quarter = quarter_value.strip() if quarter_value else _quarter_from_date(statement_date)
            elif quarter_value:
                quarter = quarter_value.strip()
                statement_date = REQUIRED_QUARTERS[quarter]["statement_date"]
            else:
                raise ValueError(
                    f"{csv_path.name} needs either a quarter column or a statement_date/datadate/date column."
                )

            if quarter not in REQUIRED_QUARTERS:
                continue

            # WRDS Compustat exports quarterly debt values in millions of USD.
            dlcq_musd = _coerce_float(normalized["dlcq"])
            dlttq_musd = _coerce_float(normalized["dlttq"])

            selected[quarter] = QuarterlyDebtObservation(
                quarter=quarter,
                statement_date=statement_date,
                dlcq_usd=dlcq_musd * 1_000_000,
                dlttq_usd=dlttq_musd * 1_000_000,
                source="Compustat export",
                field_mapping="DLCQ = Compustat DLCQ; DLTTQ = Compustat DLTTQ",
            )

    missing_quarters = [quarter for quarter in REQUIRED_QUARTERS if quarter not in selected]
    if missing_quarters:
        missing_list = ", ".join(missing_quarters)
        raise ValueError(f"{csv_path.name} does not contain all required quarters: {missing_list}")

    return [selected[quarter] for quarter in REQUIRED_QUARTERS]


def choose_question1_source(project_root: Path) -> list[QuarterlyDebtObservation]:
    compustat_path = project_root / "data" / "raw" / COMPUSTAT_INPUT_NAME
    if not compustat_path.exists():
        raise FileNotFoundError(
            "Missing data/raw/compustat_jnj_debt_quarterly.csv. "
            "Run scripts/fetch_wrds_compustat_q1.py first."
        )
    return load_compustat_observations(compustat_path)


def interpolate_daily_default_point(
    observations: list[QuarterlyDebtObservation],
) -> list[dict[str, str | float]]:
    daily_rows: list[dict[str, str | float]] = []
    for start, end in zip(observations, observations[1:]):
        span_days = (end.statement_date - start.statement_date).days
        if span_days <= 0:
            raise ValueError("Quarter observations must be strictly increasing in time.")

        for offset in range(span_days):
            current_date = start.statement_date + timedelta(days=offset)
            weight = offset / span_days
            l_usd = start.l_usd + weight * (end.l_usd - start.l_usd)
            daily_rows.append(
                {
                    "date": current_date.isoformat(),
                    "l_usd": round(l_usd, 2),
                    "l_billion_usd": round(l_usd / 1_000_000_000, 6),
                    "interpolation_start_quarter": start.quarter,
                    "interpolation_end_quarter": end.quarter,
                }
            )

    final_observation = observations[-1]
    daily_rows.append(
        {
            "date": final_observation.statement_date.isoformat(),
            "l_usd": round(final_observation.l_usd, 2),
            "l_billion_usd": round(final_observation.l_billion_usd, 6),
            "interpolation_start_quarter": final_observation.quarter,
            "interpolation_end_quarter": final_observation.quarter,
        }
    )
    return daily_rows


def _write_csv(path: Path, fieldnames: list[str], rows: list[dict[str, str | float]]) -> None:
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def run_question1(project_root: Path) -> dict[str, str]:
    observations = choose_question1_source(project_root)
    observations.sort(key=lambda item: item.statement_date)

    quarterly_rows = [
        {
            "quarter": observation.quarter,
            "statement_date": observation.statement_date.isoformat(),
            "dlcq_usd": int(observation.dlcq_usd),
            "dlttq_usd": int(observation.dlttq_usd),
            "l_usd": int(observation.l_usd),
            "dlcq_billion_usd": round(observation.dlcq_billion_usd, 3),
            "dlttq_billion_usd": round(observation.dlttq_billion_usd, 3),
            "l_billion_usd": round(observation.l_billion_usd, 3),
            "source": observation.source,
            "field_mapping": observation.field_mapping,
        }
        for observation in observations
    ]
    daily_rows = interpolate_daily_default_point(observations)

    raw_output = project_root / "data" / "raw" / "question1_jnj_quarterly_debt_series.csv"
    processed_output = project_root / "data" / "processed" / "question1_jnj_daily_default_point.csv"

    _write_csv(
        raw_output,
        [
            "quarter",
            "statement_date",
            "dlcq_usd",
            "dlttq_usd",
            "l_usd",
            "dlcq_billion_usd",
            "dlttq_billion_usd",
            "l_billion_usd",
            "source",
            "field_mapping",
        ],
        quarterly_rows,
    )
    _write_csv(
        processed_output,
        [
            "date",
            "l_usd",
            "l_billion_usd",
            "interpolation_start_quarter",
            "interpolation_end_quarter",
        ],
        daily_rows,
    )

    return {
        "source": observations[0].source,
        "quarterly_output": str(raw_output),
        "daily_output": str(processed_output),
    }


### File: `src/fixed_income_tp2/part_one.py`

In [ ]:
from __future__ import annotations

import io
import json
import math
import urllib.request
from dataclasses import dataclass
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.stats import norm

from fixed_income_tp2.question1 import run_question1


matplotlib.use("Agg")
import matplotlib.pyplot as plt


WRDS_MIN_SUPPORTED_PYTHON = (3, 8)
WRDS_MAX_SUPPORTED_PYTHON = (3, 12)
WRDS_CONFIG_PATH = Path("config/wrds.credentials.json")

PART_ONE_MARKET_CAP_PATH = Path("data/raw/part1_q2_jnj_market_cap_daily.csv")
PART_ONE_FRED_1Y_PATH = Path("data/raw/part1_q3_fred_1y_zero_coupon_daily.csv")
PART_ONE_FRED_Q5_BENCHMARK_PATH = Path("data/raw/part1_q5_fred_zero_coupon_benchmarks_daily.csv")
PART_ONE_ZERO_COUPON_PATH = Path("data/raw/part1_zero_coupon_yield_curve_daily.csv")
PART_ONE_ALIGNED_INPUTS_PATH = Path("data/processed/part1_q3_aligned_inputs.csv")
PART_ONE_ASSET_SERIES_PATH = Path("data/processed/part1_q3_asset_series.csv")
PART_ONE_ASSET_SUMMARY_PATH = Path("data/processed/part1_q3_asset_summary.csv")
PART_ONE_DEFAULT_FRONTIER_PATH = Path("data/processed/part1_q4_default_frontier.csv")
PART_ONE_SPREAD_CURVE_PATH = Path("data/processed/part1_q5_credit_spread_term_structure.csv")
PART_ONE_Q5_VALIDATION_PATH = Path("data/processed/part1_q5_risk_free_source_validation.csv")

FIGURE_Q2_PATH = Path("reports/figures/part1_q2_market_cap.png")
FIGURE_Q3_PATH = Path("reports/figures/part1_q3_asset_vs_equity.png")
FIGURE_Q4_PATH = Path("reports/figures/part1_q4_default_frontier.png")
FIGURE_Q5_PATH = Path("reports/figures/part1_q5_credit_spreads.png")

FED_NOMINAL_YIELD_CURVE_URL = "https://www.federalreserve.gov/data/yield-curve-tables/feds200628.csv"
FED_USER_AGENT = "fixed-income-tp2/1.0"
FRED_ONE_YEAR_ZERO_COUPON_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=THREEFY1"
FRED_Q5_BENCHMARK_SERIES = {
    "THREEFY1": "fred_zero_coupon_1y_pct",
    "THREEFY5": "fred_zero_coupon_5y_pct",
    "THREEFY10": "fred_zero_coupon_10y_pct",
}

TARGET_GVKEY = "006266"
TARGET_PERMCO = 21018
TARGET_PERMNO = 22111
TARGET_IID = "01"


@dataclass(frozen=True)
class AssetEstimationResult:
    iterations: int
    asset_volatility_annual: float
    asset_value_end_usd: float
    observation_count: int
    first_date: str
    last_date: str


def _load_wrds_config(project_root: Path) -> dict[str, str]:
    config_path = project_root / WRDS_CONFIG_PATH
    with config_path.open("r", encoding="utf-8") as handle:
        config = json.load(handle)
    if not config.get("wrds_username"):
        raise ValueError("config/wrds.credentials.json must include wrds_username.")
    return config


def _wrds_connection(project_root: Path):
    import sys

    current = sys.version_info[:2]
    if not (WRDS_MIN_SUPPORTED_PYTHON <= current <= WRDS_MAX_SUPPORTED_PYTHON):
        supported = ".".join(str(x) for x in WRDS_MAX_SUPPORTED_PYTHON)
        raise RuntimeError(
            "WRDS steps must run in the dedicated .venv-wrds environment. "
            f"Use Python {supported} or another supported WRDS interpreter."
        )

    try:
        import wrds
    except ImportError as exc:
        raise RuntimeError(
            "wrds is not installed in this interpreter. Activate .venv-wrds and try again."
        ) from exc

    config = _load_wrds_config(project_root)
    return wrds.Connection(wrds_username=config["wrds_username"])


def fetch_crsp_market_cap(project_root: Path) -> Path:
    output_path = project_root / PART_ONE_MARKET_CAP_PATH
    output_path.parent.mkdir(parents=True, exist_ok=True)

    query = f"""
        select
            dlycaldt as date,
            count(distinct permno) as permno_count,
            sum(abs(dlyprc) * shrout * 1000.0) as market_cap_usd,
            sum(dlycap * 1000.0) as crsp_dlycap_usd,
            sum(shrout * 1000.0) as shares_outstanding,
            avg(abs(dlyprc)) as average_price_usd
        from crsp.dsf_v2
        where permco = {TARGET_PERMCO}
          and dlycaldt between '2025-01-01' and '2025-12-31'
        group by dlycaldt
        order by dlycaldt
    """

    with _wrds_connection(project_root) as db:
        df = db.raw_sql(query, date_cols=["date"])

    if df.empty:
        raise RuntimeError("CRSP query returned no rows for Johnson & Johnson in 2025.")

    df["market_cap_usd"] = df["market_cap_usd"].astype(float)
    df["crsp_dlycap_usd"] = df["crsp_dlycap_usd"].astype(float)
    df["shares_outstanding"] = df["shares_outstanding"].astype(float)
    df["average_price_usd"] = df["average_price_usd"].astype(float)
    df.to_csv(output_path, index=False)
    return output_path


def fetch_zero_coupon_yield_curve(project_root: Path) -> Path:
    output_path = project_root / PART_ONE_ZERO_COUPON_PATH
    output_path.parent.mkdir(parents=True, exist_ok=True)

    request = urllib.request.Request(
        FED_NOMINAL_YIELD_CURVE_URL,
        headers={"User-Agent": FED_USER_AGENT},
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        text = response.read().decode("utf-8", errors="ignore")

    df = pd.read_csv(io.StringIO(text), skiprows=9)
    df["Date"] = pd.to_datetime(df["Date"])
    zero_coupon_columns = [f"SVENY{i:02d}" for i in range(1, 31)]
    keep_columns = ["Date"] + zero_coupon_columns
    df = df[keep_columns].copy()
    df = df.loc[df["Date"].between("2025-01-01", "2026-01-31")].copy()
    df.to_csv(output_path, index=False)
    return output_path


def fetch_fred_one_year_zero_coupon(project_root: Path) -> Path:
    output_path = project_root / PART_ONE_FRED_1Y_PATH
    output_path.parent.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(FRED_ONE_YEAR_ZERO_COUPON_URL)
    df = df.rename(columns={"observation_date": "date", "THREEFY1": "fred_zero_coupon_1y_pct"})
    df["date"] = pd.to_datetime(df["date"])
    df["fred_zero_coupon_1y_pct"] = pd.to_numeric(df["fred_zero_coupon_1y_pct"], errors="coerce")
    df = df.loc[df["date"].between("2025-01-01", "2026-01-31")].copy()
    df.to_csv(output_path, index=False)
    return output_path


def fetch_fred_q5_benchmarks(project_root: Path) -> Path:
    output_path = project_root / PART_ONE_FRED_Q5_BENCHMARK_PATH
    output_path.parent.mkdir(parents=True, exist_ok=True)

    merged = None
    for series_id, renamed_column in FRED_Q5_BENCHMARK_SERIES.items():
        df = pd.read_csv(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}")
        df = df.rename(columns={"observation_date": "date", series_id: renamed_column})
        df["date"] = pd.to_datetime(df["date"])
        df[renamed_column] = pd.to_numeric(df[renamed_column], errors="coerce")
        df = df.loc[df["date"].between("2025-01-01", "2026-01-31")].copy()

        if merged is None:
            merged = df
        else:
            merged = merged.merge(df, on="date", how="outer")

    if merged is None:
        raise RuntimeError("Failed to build the FRED benchmark zero-coupon series for Question 5.")

    merged = merged.sort_values("date")
    merged.to_csv(output_path, index=False)
    return output_path


def build_q3_inputs(project_root: Path) -> pd.DataFrame:
    q1_daily = pd.read_csv(project_root / "data/processed/question1_jnj_daily_default_point.csv")
    q2_market_cap = pd.read_csv(project_root / PART_ONE_MARKET_CAP_PATH)
    fred_one_year = pd.read_csv(project_root / PART_ONE_FRED_1Y_PATH)

    q1_daily["date"] = pd.to_datetime(q1_daily["date"])
    q2_market_cap["date"] = pd.to_datetime(q2_market_cap["date"])
    fred_one_year["date"] = pd.to_datetime(fred_one_year["date"])

    fred_one_year = fred_one_year[["date", "fred_zero_coupon_1y_pct"]].dropna()

    aligned = q2_market_cap.merge(fred_one_year, on="date", how="inner")
    aligned = aligned.merge(q1_daily[["date", "l_usd"]], on="date", how="inner")
    aligned = aligned.rename(
        columns={
            "market_cap_usd": "equity_value_usd",
            "l_usd": "default_point_usd",
            "fred_zero_coupon_1y_pct": "risk_free_zero_coupon_1y_pct",
        }
    )
    aligned["risk_free_zero_coupon_1y"] = aligned["risk_free_zero_coupon_1y_pct"] / 100.0
    aligned = aligned[
        [
            "date",
            "permno_count",
            "equity_value_usd",
            "crsp_dlycap_usd",
            "shares_outstanding",
            "average_price_usd",
            "default_point_usd",
            "risk_free_zero_coupon_1y_pct",
            "risk_free_zero_coupon_1y",
        ]
    ].copy()
    aligned.to_csv(project_root / PART_ONE_ALIGNED_INPUTS_PATH, index=False)
    return aligned


def _merton_equity_value(asset_value: float, default_point: float, rate: float, sigma_a: float, horizon: float) -> float:
    sigma_sqrt_t = sigma_a * math.sqrt(horizon)
    d1 = (math.log(asset_value / default_point) + (rate + 0.5 * sigma_a**2) * horizon) / sigma_sqrt_t
    d2 = d1 - sigma_sqrt_t
    return asset_value * norm.cdf(d1) - default_point * math.exp(-rate * horizon) * norm.cdf(d2)


def _solve_asset_value(equity_value: float, default_point: float, rate: float, sigma_a: float, horizon: float) -> float:
    sigma_a = max(sigma_a, 1e-6)

    def objective(asset_value: float) -> float:
        return _merton_equity_value(asset_value, default_point, rate, sigma_a, horizon) - equity_value

    lower = max(equity_value, 1.0)
    upper = max(equity_value + default_point, lower * 2.0)
    f_lower = objective(lower)
    f_upper = objective(upper)

    expansion_count = 0
    while f_upper <= 0:
        upper *= 2.0
        f_upper = objective(upper)
        expansion_count += 1
        if expansion_count > 50:
            raise RuntimeError("Failed to bracket the asset value root in the Merton step.")

    if f_lower > 0:
        lower = max(lower * 0.5, 1.0)
        f_lower = objective(lower)

    return brentq(objective, lower, upper, maxiter=200)


def estimate_asset_process(aligned_inputs: pd.DataFrame) -> tuple[pd.DataFrame, AssetEstimationResult]:
    horizon = 1.0
    tolerance = 1e-6
    max_iterations = 100

    aligned = aligned_inputs.copy()
    equity_returns = np.log(aligned["equity_value_usd"]).diff().dropna()
    sigma_a = float(equity_returns.std(ddof=1) * np.sqrt(252))
    sigma_a = max(sigma_a, 0.05)

    asset_values = None
    for iteration in range(1, max_iterations + 1):
        solved_values = []
        for row in aligned.itertuples(index=False):
            solved_value = _solve_asset_value(
                equity_value=float(row.equity_value_usd),
                default_point=float(row.default_point_usd),
                rate=float(row.risk_free_zero_coupon_1y),
                sigma_a=sigma_a,
                horizon=horizon,
            )
            solved_values.append(solved_value)

        asset_values = pd.Series(solved_values, index=aligned.index, name="asset_value_usd")
        asset_returns = np.log(asset_values).diff().dropna()
        sigma_new = float(asset_returns.std(ddof=1) * np.sqrt(252))
        if abs(sigma_new - sigma_a) < tolerance:
            sigma_a = sigma_new
            break
        sigma_a = sigma_new

    if asset_values is None:
        raise RuntimeError("Asset value iteration did not produce a valid series.")

    aligned["asset_value_usd"] = asset_values
    aligned["asset_return_log"] = np.log(aligned["asset_value_usd"]).diff()

    result = AssetEstimationResult(
        iterations=iteration,
        asset_volatility_annual=sigma_a,
        asset_value_end_usd=float(aligned.iloc[-1]["asset_value_usd"]),
        observation_count=len(aligned),
        first_date=aligned.iloc[0]["date"].strftime("%Y-%m-%d"),
        last_date=aligned.iloc[-1]["date"].strftime("%Y-%m-%d"),
    )
    return aligned, result


def build_default_frontier(project_root: Path) -> pd.DataFrame:
    q1_quarterly = pd.read_csv(project_root / "data/raw/question1_jnj_quarterly_debt_series.csv")
    end_2025 = q1_quarterly.loc[q1_quarterly["quarter"] == "2025Q4"].iloc[0]

    one_year_default_point = float(end_2025["l_usd"])
    twenty_year_default_point = float(end_2025["dlttq_usd"])
    slope = (twenty_year_default_point - one_year_default_point) / (20.0 - 1.0)

    maturities = np.arange(0.0, 30.0 + 0.25, 0.25)
    frontier = pd.DataFrame({"maturity_years": maturities})
    frontier["default_point_usd"] = one_year_default_point + slope * (frontier["maturity_years"] - 1.0)
    frontier["default_point_billion_usd"] = frontier["default_point_usd"] / 1_000_000_000
    frontier.to_csv(project_root / PART_ONE_DEFAULT_FRONTIER_PATH, index=False)
    return frontier


def build_credit_spread_curve(
    project_root: Path,
    asset_result: AssetEstimationResult,
    frontier: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.Timestamp, pd.DataFrame]:
    zero_curve = pd.read_csv(project_root / PART_ONE_ZERO_COUPON_PATH)
    zero_curve["Date"] = pd.to_datetime(zero_curve["Date"])
    fred_benchmarks = pd.read_csv(project_root / PART_ONE_FRED_Q5_BENCHMARK_PATH)
    fred_benchmarks["date"] = pd.to_datetime(fred_benchmarks["date"])

    first_trading_day_2026 = zero_curve.loc[
        (zero_curve["Date"] >= pd.Timestamp("2026-01-01")) & zero_curve["SVENY01"].notna(),
        "Date",
    ].min()
    curve_row = zero_curve.loc[zero_curve["Date"] == first_trading_day_2026].iloc[0]

    maturity_nodes = np.arange(1, 31, 1, dtype=float)
    risk_free_nodes = np.array([float(curve_row[f"SVENY{i:02d}"]) / 100.0 for i in range(1, 31)])

    maturities = np.arange(1.0, 30.0 + 0.25, 0.25)
    risk_free_curve = np.interp(maturities, maturity_nodes, risk_free_nodes)
    default_point_curve = np.interp(
        maturities,
        frontier["maturity_years"].to_numpy(),
        frontier["default_point_usd"].to_numpy(),
    )

    asset_value_0 = asset_result.asset_value_end_usd
    sigma_a = asset_result.asset_volatility_annual

    risky_prices = []
    risky_yields = []
    credit_spreads = []
    for maturity, default_point, rate in zip(maturities, default_point_curve, risk_free_curve):
        sigma_sqrt_t = sigma_a * math.sqrt(maturity)
        d1 = (math.log(asset_value_0 / default_point) + (rate + 0.5 * sigma_a**2) * maturity) / sigma_sqrt_t
        d2 = d1 - sigma_sqrt_t
        risky_price = (
            asset_value_0 * norm.cdf(-d1) + default_point * math.exp(-rate * maturity) * norm.cdf(d2)
        )
        risky_yield = -math.log(risky_price / default_point) / maturity
        credit_spread = risky_yield - rate

        risky_prices.append(risky_price)
        risky_yields.append(risky_yield)
        credit_spreads.append(credit_spread)

    spread_curve = pd.DataFrame(
        {
            "maturity_years": maturities,
            "default_point_usd": default_point_curve,
            "risk_free_yield_pct": risk_free_curve * 100.0,
            "risky_yield_pct": np.array(risky_yields) * 100.0,
            "credit_spread_pct": np.array(credit_spreads) * 100.0,
            "credit_spread_bps": np.array(credit_spreads) * 10_000.0,
            "risky_zero_coupon_price": risky_prices,
        }
    )
    spread_curve.to_csv(project_root / PART_ONE_SPREAD_CURVE_PATH, index=False)

    validation_row = fred_benchmarks.loc[fred_benchmarks["date"] == first_trading_day_2026].iloc[0]
    validation = pd.DataFrame(
        [
            {
                "date": first_trading_day_2026.strftime("%Y-%m-%d"),
                "maturity_years": 1,
                "stl_fed_fred_yield_pct": float(validation_row["fred_zero_coupon_1y_pct"]),
                "board_curve_yield_pct": float(curve_row["SVENY01"]),
            },
            {
                "date": first_trading_day_2026.strftime("%Y-%m-%d"),
                "maturity_years": 5,
                "stl_fed_fred_yield_pct": float(validation_row["fred_zero_coupon_5y_pct"]),
                "board_curve_yield_pct": float(curve_row["SVENY05"]),
            },
            {
                "date": first_trading_day_2026.strftime("%Y-%m-%d"),
                "maturity_years": 10,
                "stl_fed_fred_yield_pct": float(validation_row["fred_zero_coupon_10y_pct"]),
                "board_curve_yield_pct": float(curve_row["SVENY10"]),
            },
        ]
    )
    validation["difference_bps"] = (
        (validation["stl_fed_fred_yield_pct"] - validation["board_curve_yield_pct"]) * 100.0
    )
    validation.to_csv(project_root / PART_ONE_Q5_VALIDATION_PATH, index=False)

    return spread_curve, first_trading_day_2026, validation


def _save_q2_figure(project_root: Path, market_cap: pd.DataFrame) -> None:
    figure_path = project_root / FIGURE_Q2_PATH
    figure_path.parent.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(10, 5))
    plt.plot(market_cap["date"], market_cap["market_cap_usd"] / 1_000_000_000, linewidth=1.5)
    plt.title("Johnson & Johnson Daily Market Capitalization (2025)")
    plt.ylabel("USD billions")
    plt.xlabel("Date")
    plt.tight_layout()
    plt.savefig(figure_path, dpi=180)
    plt.close()


def _save_q3_figure(project_root: Path, asset_series: pd.DataFrame) -> None:
    figure_path = project_root / FIGURE_Q3_PATH
    figure_path.parent.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(10, 5))
    plt.plot(asset_series["date"], asset_series["equity_value_usd"] / 1_000_000_000, label="Equity value")
    plt.plot(asset_series["date"], asset_series["asset_value_usd"] / 1_000_000_000, label="Estimated asset value")
    plt.title("Question 3: Equity Value vs Estimated Asset Value")
    plt.ylabel("USD billions")
    plt.xlabel("Date")
    plt.legend()
    plt.tight_layout()
    plt.savefig(figure_path, dpi=180)
    plt.close()


def _save_q4_figure(project_root: Path, frontier: pd.DataFrame) -> None:
    figure_path = project_root / FIGURE_Q4_PATH
    figure_path.parent.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(9, 5))
    plt.plot(frontier["maturity_years"], frontier["default_point_billion_usd"], linewidth=1.6)
    plt.title("Question 4: Default Frontier")
    plt.ylabel("Default point (USD billions)")
    plt.xlabel("Maturity (years)")
    plt.tight_layout()
    plt.savefig(figure_path, dpi=180)
    plt.close()


def _save_q5_figure(project_root: Path, spread_curve: pd.DataFrame) -> None:
    figure_path = project_root / FIGURE_Q5_PATH
    figure_path.parent.mkdir(parents=True, exist_ok=True)

    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.plot(spread_curve["maturity_years"], spread_curve["risk_free_yield_pct"], label="Risk-free yield")
    ax1.plot(spread_curve["maturity_years"], spread_curve["risky_yield_pct"], label="Merton risky yield")
    ax1.set_xlabel("Maturity (years)")
    ax1.set_ylabel("Yield (%)")
    ax1.legend(loc="upper left")

    ax2 = ax1.twinx()
    ax2.plot(
        spread_curve["maturity_years"],
        spread_curve["credit_spread_bps"],
        color="black",
        linestyle="--",
        label="Credit spread",
    )
    ax2.set_ylabel("Credit spread (bps)")
    ax2.legend(loc="upper right")

    plt.title("Question 5: Risk-Free Curve, Risky Curve, and Credit Spread")
    plt.tight_layout()
    plt.savefig(figure_path, dpi=180)
    plt.close()


def run_part_one(project_root: Path) -> dict[str, str]:
    run_question1(project_root)

    fetch_crsp_market_cap(project_root)
    fetch_fred_one_year_zero_coupon(project_root)
    fetch_fred_q5_benchmarks(project_root)
    fetch_zero_coupon_yield_curve(project_root)

    aligned_inputs = build_q3_inputs(project_root)
    asset_series, asset_result = estimate_asset_process(aligned_inputs)
    asset_series.to_csv(project_root / PART_ONE_ASSET_SERIES_PATH, index=False)

    asset_summary = pd.DataFrame(
        [
            {
                "iterations": asset_result.iterations,
                "asset_volatility_annual": asset_result.asset_volatility_annual,
                "asset_volatility_pct": asset_result.asset_volatility_annual * 100.0,
                "asset_value_end_usd": asset_result.asset_value_end_usd,
                "asset_value_end_billion_usd": asset_result.asset_value_end_usd / 1_000_000_000,
                "observation_count": asset_result.observation_count,
                "first_date": asset_result.first_date,
                "last_date": asset_result.last_date,
            }
        ]
    )
    asset_summary.to_csv(project_root / PART_ONE_ASSET_SUMMARY_PATH, index=False)

    frontier = build_default_frontier(project_root)
    spread_curve, first_trading_day_2026, _validation = build_credit_spread_curve(
        project_root, asset_result, frontier
    )

    market_cap = pd.read_csv(project_root / PART_ONE_MARKET_CAP_PATH, parse_dates=["date"])
    asset_series_for_plot = pd.read_csv(project_root / PART_ONE_ASSET_SERIES_PATH, parse_dates=["date"])
    _save_q2_figure(project_root, market_cap)
    _save_q3_figure(project_root, asset_series_for_plot)
    _save_q4_figure(project_root, frontier)
    _save_q5_figure(project_root, spread_curve)

    return {
        "market_cap_path": str(project_root / PART_ONE_MARKET_CAP_PATH),
        "fred_one_year_path": str(project_root / PART_ONE_FRED_1Y_PATH),
        "zero_coupon_path": str(project_root / PART_ONE_ZERO_COUPON_PATH),
        "aligned_inputs_path": str(project_root / PART_ONE_ALIGNED_INPUTS_PATH),
        "asset_series_path": str(project_root / PART_ONE_ASSET_SERIES_PATH),
        "asset_summary_path": str(project_root / PART_ONE_ASSET_SUMMARY_PATH),
        "default_frontier_path": str(project_root / PART_ONE_DEFAULT_FRONTIER_PATH),
        "spread_curve_path": str(project_root / PART_ONE_SPREAD_CURVE_PATH),
        "q5_validation_path": str(project_root / PART_ONE_Q5_VALIDATION_PATH),
        "first_trading_day_2026": first_trading_day_2026.strftime("%Y-%m-%d"),
    }


### File: `src/fixed_income_tp2/part_two.py`

In [ ]:
from __future__ import annotations

import io
import math
import urllib.request
from dataclasses import dataclass
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
from scipy.optimize import brentq, differential_evolution


matplotlib.use("Agg")
import matplotlib.pyplot as plt


FED_NOMINAL_YIELD_CURVE_URL = "https://www.federalreserve.gov/data/yield-curve-tables/feds200628.csv"
FED_USER_AGENT = "fixed-income-tp2/1.0"
PART_TWO_VALUATION_DATE = pd.Timestamp("2026-02-27")

PART_TWO_NSS_PARAMS_PATH = Path("data/raw/part2_q1_nss_params_2026-02-27.csv")
PART_TWO_AA_YIELDS_PATH = Path("data/raw/part2_q1_aa_yields_observed.csv")
PART_TWO_FRED_Q1_BENCHMARK_PATH = Path("data/raw/part2_q1_fred_zero_coupon_benchmarks_daily.csv")
PART_TWO_TERM_STRUCTURES_PATH = Path("data/processed/part2_q1_term_structures.csv")
PART_TWO_Q1_VALIDATION_PATH = Path("data/processed/part2_q1_source_validation.csv")
PART_TWO_RISK_FREE_FIT_PATH = Path("data/processed/part2_q3_risk_free_cir_fit.csv")
PART_TWO_DUFFEE_FIT_PATH = Path("data/processed/part2_q3_duffee_fit.csv")
PART_TWO_DUFFEE_CURVE_PATH = Path("data/processed/part2_q3_duffee_curve.csv")
PART_TWO_PARAMETERS_PATH = Path("data/processed/part2_q3_parameters.csv")
PART_TWO_VALUATION_PATH = Path("data/processed/part2_q4_bond_valuation.csv")
PART_TWO_YTM_PATH = Path("data/processed/part2_q5_ytm_comparison.csv")

FIGURE_Q1_PATH = Path("reports/figures/part2_q1_term_structures.png")
FIGURE_Q3_PATH = Path("reports/figures/part2_q3_duffee_fit.png")

AA_MATURITY_YEARS = np.array([0.5, 1, 2, 3, 4, 5, 7, 10, 15, 20, 30], dtype=float)
AA_YIELD_PCT = np.array([3.76, 3.79, 3.86, 3.96, 4.08, 4.20, 4.52, 4.83, 5.42, 5.69, 6.01], dtype=float)
FRED_Q1_BENCHMARK_SERIES = {
    "THREEFY1": "fred_zero_coupon_1y_pct",
    "THREEFY5": "fred_zero_coupon_5y_pct",
    "THREEFY10": "fred_zero_coupon_10y_pct",
}


@dataclass(frozen=True)
class RiskFreeCirCalibration:
    r0: float
    kappa_r: float
    theta_r: float
    sigma_r: float
    rmse_bps: float


@dataclass(frozen=True)
class DuffeeCalibration:
    s0: float
    kappa_s: float
    theta_s: float
    sigma_s: float
    alpha: float
    beta: float
    rmse_bps: float


@dataclass(frozen=True)
class BondValuationResult:
    callable_price: float
    noncallable_price_closed_form: float
    noncallable_price_fd: float
    callable_ytm: float
    noncallable_ytm: float
    price_difference_usd: float
    ytm_difference_bps: float
    grid_r_points: int
    grid_s_points: int
    time_steps: int
    dt_years: float
    r_max: float
    s_max: float


def _mkdir_for(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def _cir_a_b(maturity_years: np.ndarray | float, kappa: float, theta: float, sigma: float) -> tuple[np.ndarray, np.ndarray]:
    maturity = np.asarray(maturity_years, dtype=float)
    a = np.ones_like(maturity)
    b = np.zeros_like(maturity)

    mask = maturity > 1e-12
    if not np.any(mask):
        return a, b

    t = maturity[mask]
    gamma = math.sqrt(kappa * kappa + 2.0 * sigma * sigma)
    exp_gamma_t = np.exp(gamma * t)
    denominator = (gamma + kappa) * (exp_gamma_t - 1.0) + 2.0 * gamma

    b[mask] = 2.0 * (exp_gamma_t - 1.0) / denominator
    a[mask] = (
        2.0 * gamma * np.exp((kappa + gamma) * t / 2.0) / denominator
    ) ** (2.0 * kappa * theta / (sigma * sigma))
    return a, b


def cir_zero_coupon_price(
    maturity_years: np.ndarray | float,
    x0: float,
    kappa: float,
    theta: float,
    sigma: float,
) -> np.ndarray:
    maturity = np.asarray(maturity_years, dtype=float)
    a, b = _cir_a_b(maturity, kappa, theta, sigma)
    return a * np.exp(-b * x0)


def cir_zero_coupon_yield(
    maturity_years: np.ndarray | float,
    x0: float,
    kappa: float,
    theta: float,
    sigma: float,
) -> np.ndarray:
    maturity = np.asarray(maturity_years, dtype=float)
    yields = np.empty_like(maturity)

    near_zero_mask = maturity <= 1e-12
    yields[near_zero_mask] = x0

    positive_mask = ~near_zero_mask
    if np.any(positive_mask):
        positive_t = maturity[positive_mask]
        prices = cir_zero_coupon_price(positive_t, x0, kappa, theta, sigma)
        yields[positive_mask] = -np.log(prices) / positive_t

    return yields


def nss_zero_coupon_yield(
    maturity_years: np.ndarray | float,
    beta0: float,
    beta1: float,
    beta2: float,
    beta3: float,
    tau1: float,
    tau2: float,
) -> np.ndarray:
    maturity = np.asarray(maturity_years, dtype=float)
    x1 = maturity / tau1
    x2 = maturity / tau2

    loading1 = np.divide(1.0 - np.exp(-x1), x1, out=np.ones_like(maturity), where=np.abs(x1) > 1e-12)
    loading2 = loading1 - np.exp(-x1)
    loading3 = np.divide(1.0 - np.exp(-x2), x2, out=np.ones_like(maturity), where=np.abs(x2) > 1e-12) - np.exp(-x2)

    return (beta0 + beta1 * loading1 + beta2 * loading2 + beta3 * loading3) / 100.0


def fetch_nss_parameters(project_root: Path) -> pd.DataFrame:
    output_path = project_root / PART_TWO_NSS_PARAMS_PATH
    _mkdir_for(output_path)

    request = urllib.request.Request(
        FED_NOMINAL_YIELD_CURVE_URL,
        headers={"User-Agent": FED_USER_AGENT},
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        text = response.read().decode("utf-8", errors="ignore")

    df = pd.read_csv(io.StringIO(text), skiprows=9)
    df["Date"] = pd.to_datetime(df["Date"])

    row = df.loc[df["Date"] == PART_TWO_VALUATION_DATE]
    if row.empty:
        raise RuntimeError("The Federal Reserve nominal yield curve table does not contain 2026-02-27.")

    selected = row[["Date", "BETA0", "BETA1", "BETA2", "BETA3", "TAU1", "TAU2"]].copy()
    selected = selected.rename(columns={"Date": "date"})
    selected.to_csv(output_path, index=False)
    return selected


def save_observed_aa_yields(project_root: Path) -> pd.DataFrame:
    output_path = project_root / PART_TWO_AA_YIELDS_PATH
    _mkdir_for(output_path)

    df = pd.DataFrame(
        {
            "maturity_years": AA_MATURITY_YEARS,
            "aa_yield_pct": AA_YIELD_PCT,
            "aa_yield_decimal": AA_YIELD_PCT / 100.0,
        }
    )
    df.to_csv(output_path, index=False)
    return df


def fetch_fred_q1_benchmarks(project_root: Path) -> pd.DataFrame:
    output_path = project_root / PART_TWO_FRED_Q1_BENCHMARK_PATH
    _mkdir_for(output_path)

    merged = None
    for series_id, renamed_column in FRED_Q1_BENCHMARK_SERIES.items():
        df = pd.read_csv(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}")
        df = df.rename(columns={"observation_date": "date", series_id: renamed_column})
        df["date"] = pd.to_datetime(df["date"])
        df[renamed_column] = pd.to_numeric(df[renamed_column], errors="coerce")
        df = df.loc[df["date"].between("2026-02-01", "2026-02-28")].copy()

        if merged is None:
            merged = df
        else:
            merged = merged.merge(df, on="date", how="outer")

    if merged is None:
        raise RuntimeError("Failed to download the Part Two FRED benchmark zero-coupon series.")

    merged = merged.sort_values("date")
    merged.to_csv(output_path, index=False)
    return merged


def build_q1_term_structures(
    project_root: Path,
    nss_parameters: pd.DataFrame,
    observed_aa_yields: pd.DataFrame,
) -> pd.DataFrame:
    output_path = project_root / PART_TWO_TERM_STRUCTURES_PATH
    _mkdir_for(output_path)

    params = nss_parameters.iloc[0]
    maturities = observed_aa_yields["maturity_years"].to_numpy(dtype=float)
    risk_free_decimal = nss_zero_coupon_yield(
        maturities,
        beta0=float(params["BETA0"]),
        beta1=float(params["BETA1"]),
        beta2=float(params["BETA2"]),
        beta3=float(params["BETA3"]),
        tau1=float(params["TAU1"]),
        tau2=float(params["TAU2"]),
    )

    result = observed_aa_yields.copy()
    result["valuation_date"] = PART_TWO_VALUATION_DATE.strftime("%Y-%m-%d")
    result["risk_free_yield_decimal"] = risk_free_decimal
    result["risk_free_yield_pct"] = risk_free_decimal * 100.0
    result["credit_spread_decimal"] = result["aa_yield_decimal"] - result["risk_free_yield_decimal"]
    result["credit_spread_pct"] = result["credit_spread_decimal"] * 100.0
    result["credit_spread_bps"] = result["credit_spread_decimal"] * 10_000.0
    result = result[
        [
            "valuation_date",
            "maturity_years",
            "risk_free_yield_pct",
            "aa_yield_pct",
            "credit_spread_pct",
            "credit_spread_bps",
            "risk_free_yield_decimal",
            "aa_yield_decimal",
            "credit_spread_decimal",
        ]
    ]
    result.to_csv(output_path, index=False)
    return result


def build_q1_source_validation(project_root: Path, term_structures: pd.DataFrame, fred_benchmarks: pd.DataFrame) -> pd.DataFrame:
    output_path = project_root / PART_TWO_Q1_VALIDATION_PATH
    _mkdir_for(output_path)

    benchmark_row = fred_benchmarks.loc[fred_benchmarks["date"] == PART_TWO_VALUATION_DATE]
    if benchmark_row.empty:
        raise RuntimeError("The FRED benchmark series do not contain 2026-02-27.")

    row = benchmark_row.iloc[0]
    term_lookup = term_structures.set_index("maturity_years")
    validation = pd.DataFrame(
        [
            {
                "date": PART_TWO_VALUATION_DATE.strftime("%Y-%m-%d"),
                "maturity_years": 1.0,
                "stl_fed_fred_yield_pct": float(row["fred_zero_coupon_1y_pct"]),
                "board_nss_yield_pct": float(term_lookup.loc[1.0, "risk_free_yield_pct"]),
            },
            {
                "date": PART_TWO_VALUATION_DATE.strftime("%Y-%m-%d"),
                "maturity_years": 5.0,
                "stl_fed_fred_yield_pct": float(row["fred_zero_coupon_5y_pct"]),
                "board_nss_yield_pct": float(term_lookup.loc[5.0, "risk_free_yield_pct"]),
            },
            {
                "date": PART_TWO_VALUATION_DATE.strftime("%Y-%m-%d"),
                "maturity_years": 10.0,
                "stl_fed_fred_yield_pct": float(row["fred_zero_coupon_10y_pct"]),
                "board_nss_yield_pct": float(term_lookup.loc[10.0, "risk_free_yield_pct"]),
            },
        ]
    )
    validation["difference_bps"] = (
        (validation["stl_fed_fred_yield_pct"] - validation["board_nss_yield_pct"]) * 100.0
    )
    validation.to_csv(output_path, index=False)
    return validation


def _feller_penalty(kappa: float, theta: float, sigma: float) -> float:
    gap = sigma * sigma - 2.0 * kappa * theta
    if gap <= 0.0:
        return 0.0
    return 100.0 * gap * gap


def calibrate_risk_free_cir(project_root: Path, term_structures: pd.DataFrame) -> tuple[pd.DataFrame, RiskFreeCirCalibration]:
    output_path = project_root / PART_TWO_RISK_FREE_FIT_PATH
    _mkdir_for(output_path)

    maturities = term_structures["maturity_years"].to_numpy(dtype=float)
    observed = term_structures["risk_free_yield_decimal"].to_numpy(dtype=float)

    def objective(params: np.ndarray) -> float:
        r0, kappa_r, theta_r, sigma_r = params
        fitted = cir_zero_coupon_yield(maturities, r0, kappa_r, theta_r, sigma_r)
        mse = float(np.mean((fitted - observed) ** 2))
        return mse + _feller_penalty(kappa_r, theta_r, sigma_r)

    bounds = [
        (0.01, 0.08),
        (0.02, 2.0),
        (0.03, 0.12),
        (0.005, 0.10),
    ]
    result = differential_evolution(objective, bounds=bounds, seed=2, maxiter=300, polish=False)
    r0, kappa_r, theta_r, sigma_r = result.x

    fitted = cir_zero_coupon_yield(maturities, r0, kappa_r, theta_r, sigma_r)
    fit_df = pd.DataFrame(
        {
            "maturity_years": maturities,
            "observed_risk_free_yield_pct": observed * 100.0,
            "fitted_risk_free_yield_pct": fitted * 100.0,
            "error_bps": (fitted - observed) * 10_000.0,
        }
    )
    fit_df.to_csv(output_path, index=False)

    rmse_bps = float(np.sqrt(np.mean((fitted - observed) ** 2)) * 10_000.0)
    calibration = RiskFreeCirCalibration(
        r0=float(r0),
        kappa_r=float(kappa_r),
        theta_r=float(theta_r),
        sigma_r=float(sigma_r),
        rmse_bps=rmse_bps,
    )
    return fit_df, calibration


def defaultable_zero_coupon_price(
    maturity_years: np.ndarray | float,
    risk_free: RiskFreeCirCalibration,
    duffee: DuffeeCalibration,
) -> np.ndarray:
    maturity = np.asarray(maturity_years, dtype=float)
    transformed_short_rate = (1.0 + duffee.beta) * risk_free.r0
    transformed_theta = (1.0 + duffee.beta) * risk_free.theta_r
    transformed_sigma = risk_free.sigma_r * math.sqrt(1.0 + duffee.beta)

    risk_free_component = cir_zero_coupon_price(
        maturity,
        transformed_short_rate,
        risk_free.kappa_r,
        transformed_theta,
        transformed_sigma,
    )
    spread_component = cir_zero_coupon_price(
        maturity,
        duffee.s0,
        duffee.kappa_s,
        duffee.theta_s,
        duffee.sigma_s,
    )
    return np.exp(-duffee.alpha * maturity) * risk_free_component * spread_component


def defaultable_zero_coupon_yield(
    maturity_years: np.ndarray | float,
    risk_free: RiskFreeCirCalibration,
    duffee: DuffeeCalibration,
) -> np.ndarray:
    maturity = np.asarray(maturity_years, dtype=float)
    yields = np.empty_like(maturity)

    near_zero_mask = maturity <= 1e-12
    yields[near_zero_mask] = duffee.alpha + (1.0 + duffee.beta) * risk_free.r0 + duffee.s0

    positive_mask = ~near_zero_mask
    if np.any(positive_mask):
        t = maturity[positive_mask]
        prices = defaultable_zero_coupon_price(t, risk_free, duffee)
        yields[positive_mask] = -np.log(prices) / t

    return yields


def calibrate_duffee_model(
    project_root: Path,
    term_structures: pd.DataFrame,
    risk_free: RiskFreeCirCalibration,
) -> tuple[pd.DataFrame, pd.DataFrame, DuffeeCalibration]:
    fit_output_path = project_root / PART_TWO_DUFFEE_FIT_PATH
    curve_output_path = project_root / PART_TWO_DUFFEE_CURVE_PATH
    parameters_output_path = project_root / PART_TWO_PARAMETERS_PATH
    _mkdir_for(fit_output_path)
    _mkdir_for(curve_output_path)
    _mkdir_for(parameters_output_path)

    maturities = term_structures["maturity_years"].to_numpy(dtype=float)
    observed_aa = term_structures["aa_yield_decimal"].to_numpy(dtype=float)

    def objective(params: np.ndarray) -> float:
        s0, kappa_s, theta_s, sigma_s, alpha, beta = params
        candidate = DuffeeCalibration(
            s0=float(s0),
            kappa_s=float(kappa_s),
            theta_s=float(theta_s),
            sigma_s=float(sigma_s),
            alpha=float(alpha),
            beta=float(beta),
            rmse_bps=0.0,
        )
        fitted = defaultable_zero_coupon_yield(maturities, risk_free, candidate)
        mse = float(np.mean((fitted - observed_aa) ** 2))
        return mse + _feller_penalty(kappa_s, theta_s, sigma_s)

    bounds = [
        (1e-6, 0.10),
        (0.001, 10.0),
        (1e-6, 0.10),
        (0.0001, 0.50),
        (0.0, 0.05),
        (0.0, 0.20),
    ]
    result = differential_evolution(objective, bounds=bounds, seed=2, maxiter=300, polish=False)
    s0, kappa_s, theta_s, sigma_s, alpha, beta = result.x

    fitted_duffee = DuffeeCalibration(
        s0=float(s0),
        kappa_s=float(kappa_s),
        theta_s=float(theta_s),
        sigma_s=float(sigma_s),
        alpha=float(alpha),
        beta=float(beta),
        rmse_bps=0.0,
    )

    fitted_aa = defaultable_zero_coupon_yield(maturities, risk_free, fitted_duffee)
    fitted_risk_free = cir_zero_coupon_yield(
        maturities,
        risk_free.r0,
        risk_free.kappa_r,
        risk_free.theta_r,
        risk_free.sigma_r,
    )
    rmse_bps = float(np.sqrt(np.mean((fitted_aa - observed_aa) ** 2)) * 10_000.0)
    fitted_duffee = DuffeeCalibration(
        s0=fitted_duffee.s0,
        kappa_s=fitted_duffee.kappa_s,
        theta_s=fitted_duffee.theta_s,
        sigma_s=fitted_duffee.sigma_s,
        alpha=fitted_duffee.alpha,
        beta=fitted_duffee.beta,
        rmse_bps=rmse_bps,
    )

    fit_df = pd.DataFrame(
        {
            "maturity_years": maturities,
            "observed_aa_yield_pct": observed_aa * 100.0,
            "fitted_aa_yield_pct": fitted_aa * 100.0,
            "observed_credit_spread_bps": (observed_aa - term_structures["risk_free_yield_decimal"].to_numpy(dtype=float)) * 10_000.0,
            "fitted_credit_spread_bps": (fitted_aa - fitted_risk_free) * 10_000.0,
            "error_bps": (fitted_aa - observed_aa) * 10_000.0,
        }
    )
    fit_df.to_csv(fit_output_path, index=False)

    curve_grid = np.linspace(0.0, 30.0, 601)
    continuous_curve = pd.DataFrame(
        {
            "maturity_years": curve_grid,
            "fitted_risk_free_yield_pct": cir_zero_coupon_yield(
                curve_grid,
                risk_free.r0,
                risk_free.kappa_r,
                risk_free.theta_r,
                risk_free.sigma_r,
            )
            * 100.0,
            "fitted_duffee_yield_pct": defaultable_zero_coupon_yield(curve_grid, risk_free, fitted_duffee) * 100.0,
        }
    )
    continuous_curve["fitted_credit_spread_bps"] = (
        continuous_curve["fitted_duffee_yield_pct"] - continuous_curve["fitted_risk_free_yield_pct"]
    ) * 100.0
    continuous_curve.to_csv(curve_output_path, index=False)

    parameter_df = pd.DataFrame(
        [
            {
                "valuation_date": PART_TWO_VALUATION_DATE.strftime("%Y-%m-%d"),
                "risk_free_r0": risk_free.r0,
                "risk_free_kappa_r": risk_free.kappa_r,
                "risk_free_theta_r": risk_free.theta_r,
                "risk_free_sigma_r": risk_free.sigma_r,
                "risk_free_rmse_bps": risk_free.rmse_bps,
                "duffee_s0": fitted_duffee.s0,
                "duffee_kappa_s": fitted_duffee.kappa_s,
                "duffee_theta_s": fitted_duffee.theta_s,
                "duffee_sigma_s": fitted_duffee.sigma_s,
                "duffee_alpha": fitted_duffee.alpha,
                "duffee_beta": fitted_duffee.beta,
                "duffee_rmse_bps": fitted_duffee.rmse_bps,
            }
        ]
    )
    parameter_df.to_csv(parameters_output_path, index=False)

    return fit_df, continuous_curve, fitted_duffee


def price_noncallable_bond_closed_form(
    risk_free: RiskFreeCirCalibration,
    duffee: DuffeeCalibration,
    face_value: float,
    coupon_rate: float,
    maturity_years: float,
    coupon_frequency: int,
) -> float:
    coupon_amount = face_value * coupon_rate / coupon_frequency
    payment_times = np.arange(1, int(round(maturity_years * coupon_frequency)) + 1, dtype=float) / coupon_frequency
    discount_factors = defaultable_zero_coupon_price(payment_times, risk_free, duffee)
    return float(np.sum(coupon_amount * discount_factors) + face_value * discount_factors[-1])


def _bilinear_grid_value(
    grid_values: np.ndarray,
    r_value: float,
    s_value: float,
    dr: float,
    ds: float,
) -> float:
    nr, ns = grid_values.shape
    r_position = float(np.clip(r_value / dr, 0.0, nr - 1))
    s_position = float(np.clip(s_value / ds, 0.0, ns - 1))

    i0 = int(math.floor(r_position))
    j0 = int(math.floor(s_position))
    i1 = min(i0 + 1, nr - 1)
    j1 = min(j0 + 1, ns - 1)

    wr = r_position - i0
    ws = s_position - j0

    return float(
        (1.0 - wr) * (1.0 - ws) * grid_values[i0, j0]
        + wr * (1.0 - ws) * grid_values[i1, j0]
        + (1.0 - wr) * ws * grid_values[i0, j1]
        + wr * ws * grid_values[i1, j1]
    )


def price_callable_bond_explicit_fd(
    risk_free: RiskFreeCirCalibration,
    duffee: DuffeeCalibration,
    face_value: float,
    coupon_rate: float,
    maturity_years: float,
    coupon_frequency: int,
    nr: int = 81,
    ns: int = 81,
    r_max: float = 0.20,
    s_max: float = 0.20,
    stability_fraction: float = 0.20,
) -> tuple[float, float, int, float]:
    coupon_amount = face_value * coupon_rate / coupon_frequency
    payment_times = np.arange(1, int(round(maturity_years * coupon_frequency)) + 1, dtype=float) / coupon_frequency

    dr = r_max / (nr - 1)
    ds = s_max / (ns - 1)
    r_grid = np.linspace(0.0, r_max, nr)
    s_grid = np.linspace(0.0, s_max, ns)
    r_mesh, s_mesh = np.meshgrid(r_grid, s_grid, indexing="ij")

    variance_r = (risk_free.sigma_r**2) * np.maximum(r_mesh, 0.0)
    variance_s = (duffee.sigma_s**2) * np.maximum(s_mesh, 0.0)
    discount_rate = duffee.alpha + (1.0 + duffee.beta) * r_mesh + s_mesh

    stability_scale = float(np.max(variance_r / (dr * dr) + variance_s / (ds * ds) + discount_rate))
    dt = stability_fraction / stability_scale
    time_steps = int(math.ceil(maturity_years / dt))
    dt = maturity_years / time_steps

    coupon_step_indices = {
        int(round(payment_time / dt))
        for payment_time in payment_times[:-1]
    }

    def apply_boundaries(values: np.ndarray, callable_flag: bool, call_cap_value: float) -> np.ndarray:
        if callable_flag:
            values[0, :] = np.minimum(call_cap_value, np.maximum(0.0, 2.0 * values[1, :] - values[2, :]))
            values[:, 0] = np.minimum(call_cap_value, np.maximum(0.0, 2.0 * values[:, 1] - values[:, 2]))
            values[0, 0] = min(call_cap_value, max(0.0, 0.5 * (values[1, 0] + values[0, 1])))
        else:
            values[0, :] = np.maximum(0.0, 2.0 * values[1, :] - values[2, :])
            values[:, 0] = np.maximum(0.0, 2.0 * values[:, 1] - values[:, 2])
            values[0, 0] = max(0.0, 0.5 * (values[1, 0] + values[0, 1]))

        values[-1, :] = 0.0
        values[:, -1] = 0.0
        return values

    def solve_grid(callable_flag: bool) -> float:
        values = np.full((nr, ns), face_value + coupon_amount, dtype=float)
        values = apply_boundaries(values, callable_flag, face_value + coupon_amount)

        for step in range(time_steps - 1, -1, -1):
            updated = values.copy()
            interior = values[1:-1, 1:-1]

            d_v_dr = (values[2:, 1:-1] - values[:-2, 1:-1]) / (2.0 * dr)
            d2_v_dr2 = (values[2:, 1:-1] - 2.0 * interior + values[:-2, 1:-1]) / (dr * dr)
            d_v_ds = (values[1:-1, 2:] - values[1:-1, :-2]) / (2.0 * ds)
            d2_v_ds2 = (values[1:-1, 2:] - 2.0 * interior + values[1:-1, :-2]) / (ds * ds)

            r_state = r_mesh[1:-1, 1:-1]
            s_state = s_mesh[1:-1, 1:-1]
            drift_r = risk_free.kappa_r * (risk_free.theta_r - r_state)
            drift_s = duffee.kappa_s * (duffee.theta_s - s_state)
            variance_r_state = (risk_free.sigma_r**2) * np.maximum(r_state, 0.0)
            variance_s_state = (duffee.sigma_s**2) * np.maximum(s_state, 0.0)
            discount_state = duffee.alpha + (1.0 + duffee.beta) * r_state + s_state

            generator_values = (
                0.5 * variance_r_state * d2_v_dr2
                + drift_r * d_v_dr
                + 0.5 * variance_s_state * d2_v_ds2
                + drift_s * d_v_ds
                - discount_state * interior
            )
            updated[1:-1, 1:-1] = interior + dt * generator_values

            call_cap_value = face_value
            if step in coupon_step_indices:
                updated += coupon_amount
                call_cap_value = face_value + coupon_amount

            updated = np.maximum(updated, 0.0)
            if callable_flag:
                updated = np.minimum(updated, call_cap_value)

            values = apply_boundaries(updated, callable_flag, call_cap_value)

        return _bilinear_grid_value(values, risk_free.r0, duffee.s0, dr, ds)

    callable_price = solve_grid(callable_flag=True)
    noncallable_fd_price = solve_grid(callable_flag=False)
    return callable_price, noncallable_fd_price, time_steps, dt


def solve_bond_ytm(price: float, face_value: float, coupon_rate: float, maturity_years: float, coupon_frequency: int) -> float:
    coupon_amount = face_value * coupon_rate / coupon_frequency
    payment_times = np.arange(1, int(round(maturity_years * coupon_frequency)) + 1, dtype=float) / coupon_frequency
    cashflows = np.full(payment_times.shape, coupon_amount, dtype=float)
    cashflows[-1] += face_value

    def objective(yield_to_maturity: float) -> float:
        per_period = yield_to_maturity / coupon_frequency
        discount_factors = (1.0 + per_period) ** (coupon_frequency * payment_times)
        return float(np.sum(cashflows / discount_factors) - price)

    return float(brentq(objective, 1e-8, 0.50))


def value_callable_and_noncallable_bonds(
    project_root: Path,
    risk_free: RiskFreeCirCalibration,
    duffee: DuffeeCalibration,
) -> BondValuationResult:
    valuation_output_path = project_root / PART_TWO_VALUATION_PATH
    ytm_output_path = project_root / PART_TWO_YTM_PATH
    _mkdir_for(valuation_output_path)
    _mkdir_for(ytm_output_path)

    face_value = 1_000_000.0
    coupon_rate = 0.04
    maturity_years = 5.0
    coupon_frequency = 2

    noncallable_price_closed_form = price_noncallable_bond_closed_form(
        risk_free=risk_free,
        duffee=duffee,
        face_value=face_value,
        coupon_rate=coupon_rate,
        maturity_years=maturity_years,
        coupon_frequency=coupon_frequency,
    )
    callable_price, noncallable_price_fd, time_steps, dt = price_callable_bond_explicit_fd(
        risk_free=risk_free,
        duffee=duffee,
        face_value=face_value,
        coupon_rate=coupon_rate,
        maturity_years=maturity_years,
        coupon_frequency=coupon_frequency,
    )

    callable_ytm = solve_bond_ytm(callable_price, face_value, coupon_rate, maturity_years, coupon_frequency)
    noncallable_ytm = solve_bond_ytm(noncallable_price_closed_form, face_value, coupon_rate, maturity_years, coupon_frequency)

    result = BondValuationResult(
        callable_price=float(callable_price),
        noncallable_price_closed_form=float(noncallable_price_closed_form),
        noncallable_price_fd=float(noncallable_price_fd),
        callable_ytm=float(callable_ytm),
        noncallable_ytm=float(noncallable_ytm),
        price_difference_usd=float(noncallable_price_closed_form - callable_price),
        ytm_difference_bps=float((callable_ytm - noncallable_ytm) * 10_000.0),
        grid_r_points=81,
        grid_s_points=81,
        time_steps=time_steps,
        dt_years=float(dt),
        r_max=0.20,
        s_max=0.20,
    )

    valuation_df = pd.DataFrame(
        [
            {
                "valuation_date": PART_TWO_VALUATION_DATE.strftime("%Y-%m-%d"),
                "face_value_usd": face_value,
                "coupon_rate": coupon_rate,
                "coupon_frequency": coupon_frequency,
                "maturity_years": maturity_years,
                "callable_price_usd": result.callable_price,
                "noncallable_price_closed_form_usd": result.noncallable_price_closed_form,
                "noncallable_price_fd_usd": result.noncallable_price_fd,
                "fd_minus_closed_form_usd": result.noncallable_price_fd - result.noncallable_price_closed_form,
                "non_coupon_call_cap_usd": face_value,
                "coupon_date_call_cap_usd": face_value + (face_value * coupon_rate / coupon_frequency),
                "grid_r_points": result.grid_r_points,
                "grid_s_points": result.grid_s_points,
                "time_steps": result.time_steps,
                "dt_years": result.dt_years,
                "r_max": result.r_max,
                "s_max": result.s_max,
            }
        ]
    )
    valuation_df.to_csv(valuation_output_path, index=False)

    ytm_df = pd.DataFrame(
        [
            {
                "valuation_date": PART_TWO_VALUATION_DATE.strftime("%Y-%m-%d"),
                "callable_price_usd": result.callable_price,
                "callable_ytm_decimal": result.callable_ytm,
                "callable_ytm_pct": result.callable_ytm * 100.0,
                "noncallable_price_usd": result.noncallable_price_closed_form,
                "noncallable_ytm_decimal": result.noncallable_ytm,
                "noncallable_ytm_pct": result.noncallable_ytm * 100.0,
                "ytm_pickup_bps": result.ytm_difference_bps,
                "price_discount_usd": result.price_difference_usd,
            }
        ]
    )
    ytm_df.to_csv(ytm_output_path, index=False)
    return result


def _save_q1_figure(project_root: Path, term_structures: pd.DataFrame) -> None:
    output_path = project_root / FIGURE_Q1_PATH
    _mkdir_for(output_path)

    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.plot(
        term_structures["maturity_years"],
        term_structures["risk_free_yield_pct"],
        marker="o",
        linewidth=1.6,
        label="Risk-free zero-coupon yield",
    )
    ax1.plot(
        term_structures["maturity_years"],
        term_structures["aa_yield_pct"],
        marker="o",
        linewidth=1.6,
        label="Observed AA yield",
    )
    ax1.set_xlabel("Maturity (years)")
    ax1.set_ylabel("Yield (%)")
    ax1.legend(loc="upper left")

    ax2 = ax1.twinx()
    ax2.bar(
        term_structures["maturity_years"],
        term_structures["credit_spread_bps"],
        width=0.6,
        alpha=0.20,
        color="black",
        label="Credit spread",
    )
    ax2.set_ylabel("Credit spread (bps)")
    ax2.legend(loc="upper right")

    plt.title("Part Two Question 1: Risk-Free Curve, AA Curve, and Credit Spreads")
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close()


def _save_q3_figure(
    project_root: Path,
    term_structures: pd.DataFrame,
    continuous_curve: pd.DataFrame,
    fit_df: pd.DataFrame,
) -> None:
    output_path = project_root / FIGURE_Q3_PATH
    _mkdir_for(output_path)

    plt.figure(figsize=(10, 5))
    plt.plot(
        continuous_curve["maturity_years"],
        continuous_curve["fitted_duffee_yield_pct"],
        linewidth=1.8,
        label="Fitted Duffee AA curve",
    )
    plt.plot(
        continuous_curve["maturity_years"],
        continuous_curve["fitted_risk_free_yield_pct"],
        linewidth=1.4,
        linestyle="--",
        label="Fitted CIR risk-free curve",
    )
    plt.scatter(
        term_structures["maturity_years"],
        term_structures["aa_yield_pct"],
        s=35,
        label="Observed AA yields",
        zorder=3,
    )
    plt.scatter(
        fit_df["maturity_years"],
        fit_df["fitted_aa_yield_pct"],
        s=25,
        label="Fitted AA nodes",
        zorder=3,
    )
    plt.xlim(0.0, 30.0)
    plt.xlabel("Maturity (years)")
    plt.ylabel("Yield (%)")
    plt.title("Part Two Question 3: Duffee Calibration")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close()


def run_part_two(project_root: Path) -> dict[str, str]:
    nss_parameters = fetch_nss_parameters(project_root)
    observed_aa_yields = save_observed_aa_yields(project_root)
    fred_benchmarks = fetch_fred_q1_benchmarks(project_root)
    term_structures = build_q1_term_structures(project_root, nss_parameters, observed_aa_yields)
    _q1_validation = build_q1_source_validation(project_root, term_structures, fred_benchmarks)
    _save_q1_figure(project_root, term_structures)

    _risk_free_fit_df, risk_free = calibrate_risk_free_cir(project_root, term_structures)
    duffee_fit_df, continuous_curve, duffee = calibrate_duffee_model(project_root, term_structures, risk_free)
    _save_q3_figure(project_root, term_structures, continuous_curve, duffee_fit_df)

    bond_valuation = value_callable_and_noncallable_bonds(project_root, risk_free, duffee)

    return {
        "nss_parameters_path": str(project_root / PART_TWO_NSS_PARAMS_PATH),
        "fred_benchmark_path": str(project_root / PART_TWO_FRED_Q1_BENCHMARK_PATH),
        "term_structures_path": str(project_root / PART_TWO_TERM_STRUCTURES_PATH),
        "q1_validation_path": str(project_root / PART_TWO_Q1_VALIDATION_PATH),
        "risk_free_fit_path": str(project_root / PART_TWO_RISK_FREE_FIT_PATH),
        "duffee_fit_path": str(project_root / PART_TWO_DUFFEE_FIT_PATH),
        "duffee_curve_path": str(project_root / PART_TWO_DUFFEE_CURVE_PATH),
        "parameters_path": str(project_root / PART_TWO_PARAMETERS_PATH),
        "bond_valuation_path": str(project_root / PART_TWO_VALUATION_PATH),
        "ytm_path": str(project_root / PART_TWO_YTM_PATH),
        "callable_price_usd": f"{bond_valuation.callable_price:.2f}",
        "noncallable_price_usd": f"{bond_valuation.noncallable_price_closed_form:.2f}",
        "callable_ytm_pct": f"{bond_valuation.callable_ytm * 100.0:.4f}",
        "noncallable_ytm_pct": f"{bond_valuation.noncallable_ytm * 100.0:.4f}",
    }


## Optional Runner Cells

These cells are included for convenience if you want to regenerate outputs directly from the notebook.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
PROJECT_ROOT


In [ ]:
# Uncomment these lines one at a time if you want to rerun the workflows.
# from fixed_income_tp2.question1 import run_question1
# from fixed_income_tp2.part_one import run_part_one
# from fixed_income_tp2.part_two import run_part_two
# run_question1(PROJECT_ROOT)
# run_part_one(PROJECT_ROOT)
# run_part_two(PROJECT_ROOT)
